# Blend MLP + Transformer Predictions
Grid-searches for the optimal blend weight using flat-weighted logloss.

In [ ]:
# ── params ───────────────────────────────────────────────────────────────────
MLP_PRED_PATH         = 'notebook_outputs/mlp_gp_predictions/predictions_combined.h5'
TRANSFORMER_PRED_PATH = 'notebook_outputs/transformer_GP_predictions/predictions_combined.h5'
TEST_DATASET_NAME     = 'plasticc_test'
OUT_PATH              = 'notebook_outputs/blended_predictions/predictions_combined.h5'

GRID_STEP   = 0.05   # search weights 0.0, 0.05, 0.10, ... 1.0
                     # weight_a = weight on TRANSFORMER, (1-weight_a) = weight on MLP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dlip_plasticc.pipelines.predict import (
    read_prediction_hdf,
    blend_predictions,
    grid_search_blend_weight,
    save_blended_predictions,
    align_prediction_frames,
)
from dlip_plasticc.pipelines.score import score_flat

In [ ]:
# ── load predictions ─────────────────────────────────────────────────────────
transformer_preds = read_prediction_hdf(TRANSFORMER_PRED_PATH)
mlp_preds         = read_prediction_hdf(MLP_PRED_PATH)

print(f'Transformer predictions: {transformer_preds.shape}')
print(f'MLP predictions:         {mlp_preds.shape}')
print(f'Transformer classes: {transformer_preds.columns.tolist()}')
print(f'MLP classes:         {mlp_preds.columns.tolist()}')

In [ ]:
# ── score each model individually ────────────────────────────────────────────
transformer_score, transformer_n = score_flat(TEST_DATASET_NAME, transformer_preds)
mlp_score, mlp_n                 = score_flat(TEST_DATASET_NAME, mlp_preds)

print(f'Transformer flat logloss: {transformer_score:.5f}  (n={transformer_n:,})')
print(f'MLP         flat logloss: {mlp_score:.5f}  (n={mlp_n:,})')

In [ ]:
# ── grid search for best blend weight ────────────────────────────────────────
# weight_a = weight on transformer, (1 - weight_a) = weight on MLP
best_weight, best_score, best_blend, history = grid_search_blend_weight(
    pred_a=transformer_preds,
    pred_b=mlp_preds,
    dataset_name=TEST_DATASET_NAME,
    grid_step=GRID_STEP,
)

print(f'Best transformer weight: {best_weight:.2f}')
print(f'Best MLP weight:         {1 - best_weight:.2f}')
print(f'Best blended logloss:    {best_score:.5f}')

In [ ]:
# ── plot blend weight vs logloss ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(history['weight_a'], history['flat_logloss'], marker='o', linewidth=2)
ax.axvline(best_weight, color='red', linestyle='--', label=f'Best weight = {best_weight:.2f}')
ax.axhline(transformer_score, color='steelblue', linestyle=':', label=f'Transformer alone = {transformer_score:.4f}')
ax.axhline(mlp_score,         color='orange',    linestyle=':', label=f'MLP alone = {mlp_score:.4f}')

ax.set_xlabel('Transformer weight (1 - MLP weight)')
ax.set_ylabel('Flat-weighted logloss')
ax.set_title('Blend weight grid search')
ax.legend()
plt.tight_layout()
plt.show()

print(history.to_string(index=False))

In [ ]:
# ── save best blend ───────────────────────────────────────────────────────────
saved_path = save_blended_predictions(best_blend, OUT_PATH)
print(f'Saved blended predictions to: {saved_path}')
best_blend.head()